In [1]:
# Optotagging analysis: read a Phy-curated Kilosort 4 sorting (run on a
# concatenated behavior+optotagging recording, see notebook 2.2), split spikes
# back per session using session_boundaries.json, then classify units as
# opto-tagged using the laser events from the optotagging session.

In [ ]:
from pathlib import Path
import yaml

# ======== parameters to change ========
session_name    = "IM-1979_2026-09-04"
base_dir_output = Path(r"E:/")
tagging_probe = "probeB"
kilosort_dir  = "kilosort4"                 # KS4 results subfolder name under <processed_session>/<tagging_probe>/
unit_criteria = {"quality": ["good"]}       # column -> criterion; see src.optotagging.load_curated_sorting
# ======================================

metadata_dir  = Path.cwd().parent
metadata_path = metadata_dir / "metadata" / f"{session_name}.yaml"
with open(metadata_path) as f:
    config = yaml.safe_load(f)

processed_path = base_dir_output / config["animal"] / "ephys" / config["sessions"]["processed"]
probe_names       = config["streams"]["probes"]

# The one probe to load + analyze for optotagging (must be Phy-curated already)
if tagging_probe not in probe_names:
    raise ValueError(f"tagging_probe={tagging_probe!r} not in this session's probes {probe_names}")

print(f"processed_session = {processed_path}")
print(f"tagging_probe     = {tagging_probe}  (available: {probe_names})")

In [7]:
# ── load curated sorting + session boundaries, for tagging_probe ─────────────
import sys, json

sys.path.append('..')
from src import load_curated_sorting, split_sorting_by_session

sorting = load_curated_sorting(processed_path, tagging_probe, kilosort_dir=kilosort_dir,
                               unit_criteria=unit_criteria)

print(f"{tagging_probe}: {len(sorting.unit_ids)} units kept (unit_criteria={unit_criteria})")
display(sorting)

probeB: 406 units kept (unit_criteria={'quality': ['good']})


UnitsSelectionSorting: 406 units - 1 segments - 30.0kHz

In [8]:
from pprint import pprint

with open(processed_path / "session_boundaries.json") as f:
    session_boundaries = json.load(f)

sessions = session_boundaries["streams"][tagging_probe]["sessions"]
pprint(sessions)
sorting_by_session = split_sorting_by_session(sorting, sessions)


for s in sessions:
    print(f"  {s['session_id']:15s} samples [{s['sample_start']}, {s['sample_end']})")

[{'n_samples': 236493534,
  'sample_end': 236493534,
  'sample_start': 0,
  'session_id': 'behavior',
  'source_dat_bytes': 181627034112,
  'source_folder': 'D:D1-1-6_IM-1979\\ephys\\2026-09-04_14-20-43'},
 {'n_samples': 30650881,
  'sample_end': 267144415,
  'sample_start': 236493534,
  'session_id': 'optotagging',
  'source_dat_bytes': 23539876608,
  'source_folder': 'D:D1-1-6_IM-1979\\ephys\\2026-09-04_16-39-38'}]
  behavior        samples [0, 236493534)
  optotagging     samples [236493534, 267144415)


In [4]:
# ── sanity check: per-session firing rate for each unit ──────────────────────
import pandas as pd

fs = sorting.get_sampling_frequency()
quality_by_unit = dict(zip(sorting.unit_ids, sorting.get_property("quality")))

rows = []
for session in sessions:
    sid = session["session_id"]
    duration_s = (session["sample_end"] - session["sample_start"]) / fs
    session_sorting = sorting_by_session[sid]
    for unit_id in session_sorting.unit_ids:
        n_spikes = len(session_sorting.get_unit_spike_train(unit_id))
        rows.append({
            "unit_id": unit_id,
            "session_id": sid,
            "quality": quality_by_unit[unit_id],
            "n_spikes": n_spikes,
            "firing_rate_hz": n_spikes / duration_s,
        })
pd.DataFrame(rows).pivot(index=["unit_id", "quality"], columns="session_id",
                          values="firing_rate_hz")

,session_id,behavior,optotagging
unit_id,quality,,
2,good,1.102483,0.038172
3,good,1.006455,0.507979
7,good,0.231127,0.156602
8,good,0.257512,0.387591
9,good,2.157649,2.309885
...,...,...,...
860,good,0.240768,0.935699
867,good,0.618791,1.076641
868,good,0.055815,0.195753
